# Preprocessing Pipeline

This notebook builds the supervised-learning matrices used by later notebooks:

- loads `train.csv` and `test.csv`
- creates the 24-hour closure target from `Created Date` and `Closed Date`
- removes leakage and identifier columns
- handles missing values using parameters learned from train only
- adds temporal, geographic, count-encoding, and interaction features
- encodes remaining categoricals and returns `X_train`, `y_train`, and `X_test`


In [ ]:
# data load

from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd()

if not (project_root / "data" / "train.csv").exists():
    project_root = project_root.parent

data_dir = project_root / "data"

train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")

date_format = "%m/%d/%Y %I:%M:%S %p"

missing_drop_threshold = 0.90
max_onehot_cardinality = 15
unknown_value = "UNKNOWN"

print("train shape:", train.shape)
print("test shape:", test.shape)

In [ ]:
# 2) target, temporal features, leakage removal

def add_target(df):
    df = df.copy()

    created = pd.to_datetime(
        df["Created Date"],
        format=date_format,
        errors="coerce"
    )

    closed = pd.to_datetime(
        df["Closed Date"],
        format=date_format,
        errors="coerce"
    )

    hours_to_close = (closed - created).dt.total_seconds() / 3600

    df["target"] = (
        (hours_to_close >= 0)
        & (hours_to_close <= 24)
    ).astype(int)

    return df


def add_temporal_features(df):
    df = df.copy()

    created = pd.to_datetime(
        df["Created Date"],
        format=date_format,
        errors="coerce"
    )

    hour = created.dt.hour

    df["created_hour"] = hour
    df["created_day_of_week"] = created.dt.dayofweek
    df["created_month"] = created.dt.month
    df["created_is_weekend"] = created.dt.dayofweek.isin([5, 6]).astype(int)

    df["created_part_of_day"] = np.select(
        [
            hour.between(0, 5),
            hour.between(6, 11),
            hour.between(12, 17),
            hour.between(18, 23)
        ],
        [
            "night",
            "morning",
            "afternoon",
            "evening"
        ],
        default=unknown_value
    )

    return df


def convert_known_numeric_columns(df):
    df = df.copy()

    numeric_cols = [
        "Incident Zip",
        "Council District",
        "BBL",
        "X Coordinate (State Plane)",
        "Y Coordinate (State Plane)",
        "Latitude",
        "Longitude"
    ]

    decimal_comma_cols = {
        "Latitude",
        "Longitude"
    }

    for col in numeric_cols:
        if col in df.columns:
            cleaned = (
                df[col]
                .astype("string")
                .str.strip()
                .str.replace("\u00a0", "", regex=False)
                .str.replace(" ", "", regex=False)
            )

            if col in decimal_comma_cols:
                cleaned = cleaned.str.replace(",", ".", regex=False)
            else:
                cleaned = cleaned.str.replace(",", "", regex=False)

            df[col] = pd.to_numeric(cleaned, errors="coerce").astype(float)

    return df


def remove_leakage_and_ids(df):
    df = df.copy()

    cols_to_drop = [
        "Closed Date",
        "Created Date",
        "Unique Key",
        "Unnamed: 0",
        "Location"
    ]

    existing_cols = [
        col for col in cols_to_drop
        if col in df.columns
    ]

    return df.drop(columns=existing_cols)


train_processed = add_target(train)
train_processed = add_temporal_features(train_processed)
train_processed = convert_known_numeric_columns(train_processed)
train_processed = remove_leakage_and_ids(train_processed)

test_processed = add_temporal_features(test)
test_processed = convert_known_numeric_columns(test_processed)
test_processed = remove_leakage_and_ids(test_processed)

print(train_processed["target"].value_counts())
print(train_processed["target"].value_counts(normalize=True).round(4))

print("train processed shape:", train_processed.shape)
print("test processed shape:", test_processed.shape)


In [ ]:
# 3) missing values

def fit_missing_value_params(df):
    missing_fraction = df.isna().mean()

    dropped_cols = [
        col
        for col in missing_fraction[
            missing_fraction > missing_drop_threshold
        ].index
        if col != "target"
    ]

    remaining = df.drop(columns=dropped_cols, errors="ignore")

    numeric_data = (
        remaining
        .select_dtypes(include="number")
        .drop(columns=["target"], errors="ignore")
    )

    numeric_medians = numeric_data.median().to_dict()

    return {
        "dropped_cols": dropped_cols,
        "numeric_medians": numeric_medians,
        "categorical_fill_value": unknown_value
    }


def apply_missing_value_params(df, params):
    df = df.copy()

    df = df.drop(columns=params["dropped_cols"], errors="ignore")

    for col, median in params["numeric_medians"].items():
        if col in df.columns:
            df[col] = df[col].fillna(median)

    categorical_cols = df.select_dtypes(include=["object", "category"]).columns

    for col in categorical_cols:
        df[col] = (
            df[col]
            .fillna(params["categorical_fill_value"])
            .astype(str)
        )

    return df


missing_params = fit_missing_value_params(train_processed)

train_processed = apply_missing_value_params(train_processed, missing_params)
test_processed = apply_missing_value_params(test_processed, missing_params)

print("dropped columns:")
print(missing_params["dropped_cols"])

print("train missing values:", train_processed.isna().sum().sum())
print("test missing values:", test_processed.isna().sum().sum())

In [ ]:
# 4) Derived temporal features

def add_derived_temporal_features(df):
    df = df.copy()

    df["created_hour_sin"] = np.sin(2 * np.pi * df["created_hour"] / 24)
    df["created_hour_cos"] = np.cos(2 * np.pi * df["created_hour"] / 24)

    df["created_day_sin"] = np.sin(
        2 * np.pi * df["created_day_of_week"] / 7
    )

    df["created_day_cos"] = np.cos(
        2 * np.pi * df["created_day_of_week"] / 7
    )

    df["created_month_sin"] = np.sin(
        2 * np.pi * df["created_month"] / 12
    )

    df["created_month_cos"] = np.cos(
        2 * np.pi * df["created_month"] / 12
    )

    df["created_is_business_hour"] = (
        df["created_hour"].between(9, 17)
    ).astype(int)

    df["created_is_night"] = (
        df["created_hour"].between(0, 5)
    ).astype(int)

    return df


train_processed = add_derived_temporal_features(train_processed)
test_processed = add_derived_temporal_features(test_processed)

temporal_features = [
    "created_hour_sin",
    "created_hour_cos",
    "created_day_sin",
    "created_day_cos",
    "created_month_sin",
    "created_month_cos",
    "created_is_business_hour",
    "created_is_night"
]

print("temporal features added:", temporal_features)

In [ ]:
# 5) geographic features

def add_geographic_features(df):
    df = df.copy()

    if "Borough" in df.columns:
        borough = df["Borough"].astype(str).str.upper()

        df["is_manhattan"] = (borough == "MANHATTAN").astype(int)
        df["is_brooklyn"] = (borough == "BROOKLYN").astype(int)
        df["is_queens"] = (borough == "QUEENS").astype(int)
        df["is_bronx"] = (borough == "BRONX").astype(int)
        df["is_staten_island"] = (borough == "STATEN ISLAND").astype(int)

    if "Incident Zip" in df.columns:
        df["incident_zip_group"] = (
            df["Incident Zip"] // 100
        ).astype(int).astype(str)

    if "Latitude" in df.columns and "Longitude" in df.columns:
        nyc_latitude = 40.7128
        nyc_longitude = -74.0060

        df["distance_from_nyc_center"] = np.sqrt(
            (df["Latitude"] - nyc_latitude) ** 2
            + (df["Longitude"] - nyc_longitude) ** 2
        )

        df["latitude_longitude_interaction"] = (
            df["Latitude"] * df["Longitude"]
        )

    return df


train_processed = add_geographic_features(train_processed)
test_processed = add_geographic_features(test_processed)

geographic_features = [
    col
    for col in [
        "is_manhattan",
        "is_brooklyn",
        "is_queens",
        "is_bronx",
        "is_staten_island",
        "incident_zip_group",
        "distance_from_nyc_center",
        "latitude_longitude_interaction"
    ]
    if col in train_processed.columns
]

print("geographic features added:", geographic_features)

In [ ]:
# 6) count encoding

def fit_count_encoding(df, columns):
    count_maps = {}

    for col in columns:
        if col in df.columns:
            count_maps[col] = (
                df[col]
                .astype(str)
                .value_counts(normalize=True)
                .to_dict()
            )

    return count_maps


def apply_count_encoding(df, count_maps):
    df = df.copy()

    for col, mapping in count_maps.items():
        if col in df.columns:
            df[f"{col}_frequency"] = (
                df[col]
                .astype(str)
                .map(mapping)
                .fillna(0)
            )

    return df


count_encoding_cols = [
    "Agency",
    "Agency Name",
    "Problem (formerly Complaint Type)",
    "Borough",
    "Incident Zip",
    "incident_zip_group"
]

count_maps = fit_count_encoding(
    train_processed,
    count_encoding_cols
)

train_processed = apply_count_encoding(
    train_processed,
    count_maps
)

test_processed = apply_count_encoding(
    test_processed,
    count_maps
)

count_features = [
    f"{col}_frequency"
    for col in count_encoding_cols
    if f"{col}_frequency" in train_processed.columns
]

print("count-encoded features added:", count_features)

In [ ]:
# 7) interaction features

def add_interaction_features(df):
    df = df.copy()

    df["weekend_night_interaction"] = (
        df["created_is_weekend"] * df["created_is_night"]
    )

    df["business_hour_weekday_interaction"] = (
        df["created_is_business_hour"] * (1 - df["created_is_weekend"])
    )

    if "Borough_frequency" in df.columns:
        df["borough_weekend_interaction"] = (
            df["Borough_frequency"] * df["created_is_weekend"]
        )

    return df


train_processed = add_interaction_features(train_processed)
test_processed = add_interaction_features(test_processed)

interaction_features = [
    col
    for col in [
        "weekend_night_interaction",
        "business_hour_weekday_interaction",
        "borough_weekend_interaction"
    ]
    if col in train_processed.columns
]

print("interaction features added:", interaction_features)

In [ ]:
# 8) categorical encoding

def fit_categorical_encoding_params(df):
    categorical_cols = (
        df
        .select_dtypes(include=["object", "category"])
        .columns
        .tolist()
    )

    categorical_cols = [
        col
        for col in categorical_cols
        if col != "target"
    ]

    onehot_cols = []
    label_maps = {}

    for col in categorical_cols:
        n_unique = df[col].nunique(dropna=False)

        if n_unique <= max_onehot_cardinality:
            onehot_cols.append(col)
        else:
            categories = sorted(df[col].astype(str).unique().tolist())

            mapping = {
                category: index
                for index, category in enumerate(categories)
            }

            mapping["__UNKNOWN__"] = len(mapping)
            label_maps[col] = mapping

    return {
        "onehot_cols": onehot_cols,
        "label_maps": label_maps
    }


def apply_categorical_encoding(df, params):
    df = df.copy()

    for col, mapping in params["label_maps"].items():
        if col in df.columns:
            unknown_code = mapping["__UNKNOWN__"]

            df[col] = (
                df[col]
                .astype(str)
                .map(mapping)
                .fillna(unknown_code)
                .astype(int)
            )

    existing_onehot_cols = [
        col
        for col in params["onehot_cols"]
        if col in df.columns
    ]

    df = pd.get_dummies(
        df,
        columns=existing_onehot_cols,
        dtype=int
    )

    return df


encoding_params = fit_categorical_encoding_params(train_processed)

train_encoded = apply_categorical_encoding(
    train_processed,
    encoding_params
)

test_encoded = apply_categorical_encoding(
    test_processed,
    encoding_params
)

print("one-hot columns:")
print(encoding_params["onehot_cols"])

print("label-encoded columns:")
print(list(encoding_params["label_maps"].keys()))

print("train shape after encoding:", train_encoded.shape)
print("test shape after encoding:", test_encoded.shape)

In [ ]:
# 9) final matrices and checks

def create_final_matrices(train_df, test_df):
    feature_cols = [
        col
        for col in train_df.columns
        if col != "target"
    ]

    X_train = train_df[feature_cols].copy()
    y_train = train_df["target"].copy()

    X_test = test_df.reindex(
        columns=feature_cols,
        fill_value=0
    ).copy()

    return X_train, y_train, X_test


X_train, y_train, X_test = create_final_matrices(
    train_encoded,
    test_encoded
)

assert "target" not in X_train.columns
assert "target" not in X_test.columns
assert "Closed Date" not in X_train.columns
assert "Closed Date" not in X_test.columns
assert X_train.shape[1] == X_test.shape[1]
assert X_train.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0
assert len(X_train.select_dtypes(exclude="number").columns) == 0
assert len(X_test.select_dtypes(exclude="number").columns) == 0

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)

In [ ]:
# 10) final preprocessing summary

engineered_features = (
    temporal_features
    + geographic_features
    + count_features
    + interaction_features
)

summary = pd.DataFrame({
    "metric": [
        "training rows",
        "test rows",
        "final features",
        "engineered features tracked",
        "positive target rate",
        "missing values in X_train",
        "missing values in X_test",
        "non-numeric columns in X_train",
        "non-numeric columns in X_test"
    ],
    "value": [
        X_train.shape[0],
        X_test.shape[0],
        X_train.shape[1],
        len([col for col in engineered_features if col in train_processed.columns]),
        round(y_train.mean(), 4),
        int(X_train.isna().sum().sum()),
        int(X_test.isna().sum().sum()),
        len(X_train.select_dtypes(exclude="number").columns),
        len(X_test.select_dtypes(exclude="number").columns)
    ]
})

display(summary)

target_summary = (
    y_train
    .value_counts()
    .sort_index()
    .rename_axis("target")
    .reset_index(name="count")
)

target_summary["percentage"] = (
    target_summary["count"] / len(y_train) * 100
).round(2)

display(target_summary)

top_missing_or_constant_checks = pd.DataFrame({
    "missing_values": X_train.isna().sum(),
    "unique_values": X_train.nunique(),
    "dtype": X_train.dtypes.astype(str)
}).sort_values(["missing_values", "unique_values"], ascending=[False, True]).head(15)

display(top_missing_or_constant_checks)
